Section 1 we do setup import first by using pathlib (from pathlib import path)

In [107]:


import os
import hashlib
import pandas as pd
from pathlib import Path
from datetime import datetime
import logging

# ====================== LOGGING SETUP ======================
logging.basicConfig(level=logging.INFO, 
                    format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# ====================== PATHS  ======================
ROOT = Path.cwd()
DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
IMAGE_DIR = ROOT / "image_chart"

DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

RAW_FILE = DATA_RAW / "Data_set.csv"
METADATA_FILE = DATA_RAW / "metadata.txt"
PROCESSED_GLOBAL = DATA_PROCESSED / "global_clean.csv"
PROCESSED_MALAYSIA = DATA_PROCESSED / "malaysia_clean.csv"

logger.info("Folders and paths ready.") #looks at out oupt ,logger function help
#20....810 is asctime , and INFO is levelname(Log level), and message is u write ,which is "Floder and paths ready".

2026-03-10 17:36:45,709 - INFO - Folders and paths ready.


Section 2 check RAW_File is empty or not

In [108]:
if not RAW_FILE.exists():
    logger.error(" Data_set.csv not found!")
    print("Please download from this link:")
    print("https://www.kaggle.com/datasets/akshatsharma2/global-jobs-gdp-and-unemployment-data-19912022")
    print("Rename the CSV to 'Data_set.csv' and put it in data/raw/")

else:
    logger.info(f"Raw file found: {RAW_FILE}")

2026-03-10 17:36:45,768 - INFO - Raw file found: c:\BEGIN\Malaysia-youth-unemployment-analysis\notebook\data\raw\Data_set.csv


In [109]:
#Checksum + Metadata
def compute_checksum(file_path):
     hash_sha = hashlib.sha256()
     with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
           hash_sha.update(chunk)
           return hash_sha.hexdigest()

checksum = compute_checksum(RAW_FILE) 
file_size = RAW_FILE.stat().st_size

metadata = {
    "source":"Kaggle - Global Jobs, GDP & Unemployment Data (1991-2022)",
    "url":"https://www.kaggle.com/datasets/akshatsharma2/global-jobs-gdp-and-unemployment-data-19912022",
    "download_date": datetime.now().isoformat(),
    "file_name": RAW_FILE.name,
    "file_size_bytes": file_size,
    "checksum_sha256":checksum,
    "row_expected":5751
}

with open(METADATA_FILE,"w") as f: #with help to auto close file f=open(METADAT_FILE,"w") open(filename_mode) w=write
    for k, v in metadata.items(): #k=key v=value metadata is a dictionary inside has key and value pair ,
        # like source is key and Kagle... is value
        f.write(f"{k}:{v}\n") #f.write is to write the key and valur in the file we open in write mode
        #and \n is to write in new line,line by line for each key and value pair

logger.info(f"Checksum:{checksum}")
logger.info(f"Metadata saved to {METADATA_FILE}")
logger.info(f"File Name: {RAW_FILE.name}")


2026-03-10 17:36:45,796 - INFO - Checksum:a119f1898c4ba24ca8569c5fecda644caf94710d9f8f3a90f73883abb5dcf05c
2026-03-10 17:36:45,798 - INFO - Metadata saved to c:\BEGIN\Malaysia-youth-unemployment-analysis\notebook\data\raw\metadata.txt
2026-03-10 17:36:45,800 - INFO - File Name: Data_set.csv


In [110]:
# Load and check data validation
df = pd.read_csv(RAW_FILE)#Data_set.csv in data/raw

expected_columns = ['Country Name', 'Year', 'Employment Sector: Agriculture',
                    'Employment Sector: Industry', 'Employment Sector: Services',
                    'Unemployment Rate', 'GDP (in USD)']
#assert condition, "error message" <<<<<<<<< important to know about assert function(python function)
assert set(expected_columns).issubset(df.columns), "Missing columns!"
assert df.shape == (5751, 7), "shape must be (5751,7)"
assert (df['Unemployment Rate'] >=0 ).all() and (df['Unemployment Rate'] <=100).all(),"Invalid unemployment rate"
#becaz Unemployment Rate is in % so 0-100 only ,
#its not negative

logger.info(" All validation checks passed!")

2026-03-10 17:36:45,842 - INFO -  All validation checks passed!


In [111]:
#Create Clean Version and pull the Malaysia version dataframe
df_clean = df.copy()




df_clean['Unemployment Rate'] = df_clean['Unemployment Rate'].round(2)
df_clean['Employment Sector: Agriculture'] = df_clean['Employment Sector: Agriculture'].round(1)
df_clean['Employment Sector: Industry'] = df_clean['Employment Sector: Industry'].round(1)
df_clean['Employment Sector: Services'] = df_clean['Employment Sector: Services'].round(1)
df_clean['GDP (billion USD)'] = (df_clean['GDP (in USD)'] / 1_000_000_000).round(2)
df_clean = df_clean.drop(columns=['GDP (in USD)'])

# Malaysia subset (with your fixes)
df_malaysia = df_clean[df_clean['Country Name'] == 'Malaysia'].copy()
df_malaysia = df_malaysia.sort_values('Year').reset_index(drop=True)
df_malaysia['covid_period'] = ((df_malaysia['Year'] >= 2020) & 
                              (df_malaysia['Year'] <= 2022)).astype(int)
df_malaysia['unemp_change_pct'] = df_malaysia['Unemployment Rate'].diff().round(2)
df_malaysia['unemp_change_pct'] = df_malaysia['unemp_change_pct'].fillna(0) #fill the NaN

df_malaysia.head()

,Country Name,Year,Employment Sector: Agriculture,Employment Sector: Industry,Employment Sector: Services,Unemployment Rate,GDP (billion USD),covid_period,unemp_change_pct
0,Malaysia,1991,18.9,31.4,49.7,3.70,49.14,0,0.00
1,Malaysia,1992,18.5,31.6,49.9,3.71,59.17,0,0.01
2,Malaysia,1993,17.8,32.0,50.2,4.11,66.89,0,0.40
3,Malaysia,1994,17.5,32.2,50.3,3.64,74.48,0,-0.47
4,Malaysia,1995,17.2,32.3,50.5,3.15,88.71,0,-0.49


In [112]:
# Save Processed Files
df_clean.to_csv(PROCESSED_GLOBAL, index=False)
df_malaysia.to_csv(PROCESSED_MALAYSIA, index=False)

logger.info(f"Processed files saved:\n{PROCESSED_GLOBAL}\n{PROCESSED_MALAYSIA}") #n\ is new line for behind




2026-03-10 17:36:45,964 - INFO - Processed files saved:
c:\BEGIN\Malaysia-youth-unemployment-analysis\notebook\data\processed\global_clean.csv
c:\BEGIN\Malaysia-youth-unemployment-analysis\notebook\data\processed\malaysia_clean.csv
